# Day 4: Tract Profilometry & Tractometry with Scilpy

Welcome to the **Day 4 Interactive Tractometry Laboratory**.

In this notebook, we load the structured along-tract microstructural profiles produced by `scil_bundle_mean_std` and `scil_json_merge_entries` during **Tutorial 4.6**.

We will visualize:
1. **Along-Tract Profiles**: Mean metric values (FA, MD, NDI) evaluated at equidistant nodes along the bundle centroid.
2. **Confidence Ribbons**: Standard deviation bounds representing cross-sectional microstructural dispersion.

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

def find_file(filename):
    candidates = [filename, os.path.join("..", filename)]
    for path in candidates:
        if os.path.exists(path):
            return path
    return None

json_path = find_file("tractometry_profiles.json")
whole_json_path = find_file("tractometry_whole_bundles.json")

print(f"Tractometry Profiles JSON     : {json_path}")
print(f"Whole-Bundle Statistics JSON  : {whole_json_path}")

In [ ]:
if json_path and os.path.exists(json_path):
    with open(json_path, "r") as f:
        profiles_data = json.load(f)
    
    # Extract subject dictionary
    subject_key = list(profiles_data.keys())[0] if profiles_data else "sub-01"
    sub_dict = profiles_data.get(subject_key, profiles_data)
    
    bundles = list(sub_dict.keys())
    print(f"Available Bundles in {subject_key}:", bundles)
    
    fig, axes = plt.subplots(len(bundles), 1, figsize=(10, 4 * len(bundles)), squeeze=False)
    
    for i, bundle_name in enumerate(bundles):
        bundle_info = sub_dict[bundle_name]
        ax = axes[i, 0]
        
        # Plot each available metric profile
        for metric_name, stats in bundle_info.items():
            if isinstance(stats, dict) and "mean" in stats:
                means = np.array(stats["mean"])
                stds = np.array(stats.get("std", np.zeros_like(means)))
                nodes = np.arange(len(means))
                
                line, = ax.plot(nodes, means, label=f"{metric_name} (mean)", linewidth=2)
                ax.fill_between(nodes, means - stds, means + stds, color=line.get_color(), alpha=0.2)
        
        ax.set_title(f"Along-Tract Microstructural Profile: {bundle_name}", fontsize=13, fontweight="bold")
        ax.set_xlabel("Centroid Node (Anterior/Proximal -> Posterior/Distal)", fontsize=11)
        ax.set_ylabel("Metric Value", fontsize=11)
        ax.legend(loc="upper right", frameon=True)
        ax.grid(True, linestyle="--", alpha=0.5)
    
    plt.tight_layout()
    plt.show()
else:
    print("Tractometry profiles JSON not found yet. Please run tutorial_4.6_tracts_to_tables.sh first.")